# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

I'm picking this over the other three because the starter data shows real, sizeable spread *within the same position tier* — see Section 3. Two pages both sitting on page 1 of search results can have very different click-through rates, and that gap is exactly what this lane is built to rank. It also lines up with the kind of work I want to get good at this internship: not just detecting that something changed (Lane 1), not clustering pages into types (Lane 3), but building a ranked, evidence-backed list that a real reviewer could act on with limited time — which is closer to the retrieval/ranking and evaluation work I've been doing on my dissertation (comparing candidates against a baseline, validating with proper held-out splits, not just reporting a headline number).

I'm treating this as provisional, not final — the guide gives me until end of Week 4 to confirm or switch, and I want to see the CTR-vs-engagement split more closely and check the warehouse-scale numbers before I commit fully.

In [1]:
import pandas as pd

pd.set_option('display.width', 120)

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print('starter dataset shape:', df.shape)

starter dataset shape: (30000, 44)


## 2. The question: decision, action, cost of a wrong call

**Question:** Among pages that are already visible in search (indexed, earning impressions), which ones are under-capturing clicks or engagement *relative to what their position tier and volume would predict* — and should therefore be reviewed first?

**Decision this improves:** which handful of pages a content reviewer opens first this week, out of a much larger pool of "visible" pages, when they only have time to review a limited number.

**Who acts, and what they do:** a FlyRank content/SEO reviewer. They take the top of the ranked queue, open each page, and — depending on the reason code attached — either rewrite the title/meta description (CTR gap while position is strong) or review on-page content and layout (engagement/scroll gap despite decent traffic).

**Cost of a wrong recommendation:**
- **False positive** (page flagged as underperforming, but it isn't really): wastes a reviewer's limited hour on a page that didn't need it — and that hour is now not spent on a page that genuinely did.
- **False negative** (a genuinely underperforming, high-impression page is missed): the page keeps quietly leaking clicks or engagement every day it isn't reviewed, and nobody ever finds out, because nothing in the raw data "alerts" on its own — someone has to go look.

Because reviewer time is the scarce resource, this is fundamentally a ranking-quality problem, not an accuracy problem — precision at the top of the list (precision@K) matters far more than getting every borderline row right.

In [2]:
# No additional numbers needed for this section — the framing above is answered in words,
# per the framing-ml-problems skill ("decision, action, cost" are qualitative until Section 3
# grounds the lane choice in real data).

## 3. Quick look at the data (2-3 real numbers)

Loaded straight from `data/raw/content_refresh_anonymized.csv` (30,000 rows × 44 columns — matches the lane guide). No numbers below are invented; every one is computed in the cell beneath this markdown.

In [3]:
# Apply the same filters the starter pipeline uses (impressions_90d > 0, content_age_days >= 90),
# dedup by content_id
filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
print('rows after starter filters:', len(filtered))

# avg_position == 0 means "no data", not rank zero -- confirms I've read the data-dictionary gotcha
no_position_data = (df['avg_position'] == 0).sum()
print('rows with avg_position == 0 (no data, per data dictionary):', no_position_data)

# "Visible" pool for this lane: ranked 1-20 with meaningful impression volume
visible = filtered[(filtered['avg_position'] > 0) & (filtered['avg_position'] <= 20) & (filtered['impressions_90d'] >= 500)]
print('candidate pool (position 1-20, impressions_90d >= 500):', len(visible))
print()

# The core justification for this lane: within ONE position tier, CTR varies enormously.
# If position explained CTR fully, this spread would be tiny.
page1 = filtered[filtered['position_tier'] == 'page_1']
print(f"page_1 tier only (n={len(page1)}): CTR mean={page1['ctr'].mean():.2f}, "
      f"median={page1['ctr'].median():.2f}, std={page1['ctr'].std():.2f}, max={page1['ctr'].max():.2f}")
low_ctr_on_page1 = (page1['ctr'] < 0.2).sum()
strong_ctr_on_page1 = (page1['ctr'] > 1.0).sum()
print(f"  -> {low_ctr_on_page1} page_1 pages have ctr < 0.2, while {strong_ctr_on_page1} have ctr > 1.0")
print("  -> same tier, very different outcomes: position alone doesn't explain this")
print()

# Using the starter's own low_ctr_visible_page reason code definition
low_ctr_candidates = filtered[(filtered['impressions_90d'] >= 500) & (filtered['avg_position'] > 0)
                               & (filtered['avg_position'] <= 20) & (filtered['ctr'] < 0.5)]
pct = 100 * len(low_ctr_candidates) / len(filtered)
print(f"low_ctr_visible_page candidates (starter reason-code definition): {len(low_ctr_candidates)} "
      f"({pct:.1f}% of the filtered dataset)")

rows after starter filters: 30000
rows with avg_position == 0 (no data, per data dictionary): 1205
candidate pool (position 1-20, impressions_90d >= 500): 12023

page_1 tier only (n=11814): CTR mean=0.65, median=0.16, std=3.09, max=100.00
  -> 6520 page_1 pages have ctr < 0.2, while 1005 have ctr > 1.0
  -> same tier, very different outcomes: position alone doesn't explain this

low_ctr_visible_page candidates (starter reason-code definition): 9759 (32.5% of the filtered dataset)


## 4. Careful words: what I can and can't claim

**What this work will be able to say:** observed, position-tier-adjusted patterns in click and engagement behavior on this anonymized starter slice (and, once validated, the warehouse release); a ranked, reason-coded list of review candidates suitable for decision support, where "high in the ranking" means "worth a human look first," not "guaranteed to be a problem."

**What it will never say:** that a title or meta rewrite *caused* a CTR increase (that needs a genuine experiment, not this data); that any result reveals a Google ranking factor; that AI referral behavior means anything about "AI visibility" beyond sessions someone actually clicked into; that a page flagged here is a guaranteed win if fixed. All numbers above come from a single anonymized 30,000-row snapshot (32 clients) — they motivate the lane, they don't yet validate a model, and the 32.5% flagged by a flat threshold is itself evidence that a naive rule is too blunt for a limited-capacity review queue — which is precisely why this needs a ranked, tier-adjusted approach rather than a single cutoff.

In [4]:
# No additional numbers needed for this section -- the caveats above are qualitative,
# and are grounded in the flat-threshold result already computed in Section 3
# (9,759 candidates / 32.5% of the filtered pool from one static rule is the evidence
# that a smarter, tier-aware ranking is needed rather than a single cutoff).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.